# 04 - Multi Horizon Final Models
Notebook de produccion para entrenar y comparar modelos multi-horizonte (H1, H2, H3) con blacklist estricta y seleccionar campeones.

In [3]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

base_dir = Path.cwd()
for root in [base_dir, *base_dir.parents]:
    if (root / "src").exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        break

from src.data_processing.build_dataset import get_training_features

warnings.filterwarnings("ignore")

dataset_path = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "data" / "processed" / "dataset_entrenamiento_final.csv"
    if candidate.exists():
        dataset_path = candidate
        break
if dataset_path is None:
    raise FileNotFoundError("dataset_entrenamiento_final.csv not found under data/processed")

df = pd.read_csv(dataset_path, parse_dates=["date"])
print("dataset:", dataset_path.resolve())
print("shape:", df.shape)

blacklist = [
    "precio_provincial_lag_1",
    "precio_provincial_lag_2",
    "precio_provincial_lag_3",
    "precio_vecinos_media_lag1",
    "precio_nacional_base_ma3",
    "precio_nacional_base_ma6",
    "precio_nacional_base_vol3",
    "precio_nacional_base_vol6",
]
target_candidates = ["precio_provincial_TARGET_H1", "precio_provincial_TARGET_H2", "precio_provincial_TARGET_H3"]
missing_targets = [t for t in target_candidates if t not in df.columns]
if missing_targets:
    raise ValueError(f"Missing target columns: {missing_targets}")
available_targets = target_candidates
horizons = [1, 2, 3]

split_date = pd.Timestamp("2021-01-01")
train_mask = df["date"] < split_date
test_mask = ~train_mask

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()
print("train rows:", train_df.shape[0], "test rows:", test_df.shape[0])

dataset: C:\Users\marco\Desktop\Repos\DATAGIA-21\data\processed\dataset_entrenamiento_final.csv
shape: (7047, 78)
train rows: 5394 test rows: 1653


In [4]:
identifiers = ["date", "provincia", "cereal_predominante"]
training_cols = get_training_features(df)
feature_cols = [
    c for c in training_cols
    if c in df.columns and c not in identifiers + available_targets
]
feature_cols = [c for c in feature_cols if c not in blacklist]
print("feature count:", len(feature_cols))
print("blacklist applied:", [c for c in blacklist if c in df.columns])

X_full = df[feature_cols].copy()
bool_cols = X_full.select_dtypes(include=["bool"]).columns
if len(bool_cols) > 0:
    X_full[bool_cols] = X_full[bool_cols].astype(int)

cat_cols = X_full.select_dtypes(include=["object", "category"]).columns.tolist()
if cat_cols:
    X_full = pd.get_dummies(X_full, columns=cat_cols, drop_first=False)

X_train = X_full.loc[train_mask].copy()
X_test = X_full.loc[test_mask].copy()
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

base_price_col = "precio_provincial_lag_1"
if base_price_col not in df.columns:
    raise ValueError("precio_provincial_lag_1 missing for classification target")

def build_targets(horizon: int):
    target_reg = f"precio_provincial_TARGET_H{horizon}"
    y_train_reg = train_df[target_reg]
    y_test_reg = test_df[target_reg]
    base_train = train_df[base_price_col]
    base_test = test_df[base_price_col]
    y_train_clf = (y_train_reg - base_train > 0).astype(int)
    y_test_clf = (y_test_reg - base_test > 0).astype(int)
    return y_train_reg, y_test_reg, y_train_clf, y_test_clf

feature count: 64
blacklist applied: ['precio_provincial_lag_1', 'precio_provincial_lag_2', 'precio_provincial_lag_3', 'precio_vecinos_media_lag1', 'precio_nacional_base_ma3', 'precio_nacional_base_ma6', 'precio_nacional_base_vol3', 'precio_nacional_base_vol6']


## 3. Entrenamiento y tuning de regresores por horizonte (RF/XGB)
Para cada horizonte, ejecutar RandomizedSearchCV (20 iteraciones) sobre RandomForestRegressor y XGBRegressor con scoring Pearson.

In [5]:
tscv = TimeSeriesSplit(n_splits=5)

def pearson_scorer(estimator, X, y):
    preds = estimator.predict(X)
    if y.nunique() <= 1:
        return 0.0
    return pearsonr(y, preds)[0]

def regression_metrics(y_true, y_pred, base_series):
    aligned = pd.concat([y_true, base_series], axis=1).dropna()
    if aligned.empty:
        return {"Pearson": np.nan, "MAE": np.nan, "RMSE": np.nan, "DA": np.nan}
    y_true_clean = aligned.iloc[:, 0]
    base_clean = aligned.iloc[:, 1]
    y_pred_clean = pd.Series(y_pred, index=y_true.index).loc[aligned.index]
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    pearson = pearsonr(y_true_clean, y_pred_clean)[0] if y_true_clean.nunique() > 1 else np.nan
    da = (np.sign(y_pred_clean - base_clean) == np.sign(y_true_clean - base_clean)).mean()
    return {
        "Pearson": float(pearson) if pearson == pearson else np.nan,
        "MAE": float(mae),
        "RMSE": float(rmse),
        "DA": float(da),
    }

def top_features_from_model(model, feature_names, top_k=5):
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        order = np.argsort(importances)[::-1][:top_k]
        return [feature_names[i] for i in order]
    if hasattr(model, "coef_"):
        coefs = np.ravel(model.coef_)
        order = np.argsort(np.abs(coefs))[::-1][:top_k]
        return [feature_names[i] for i in order]
    return []

reg_param_grids = {
    "RF": {
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [4, 6, 8, 12, None],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", 0.6, 0.8],
    },
    "XGB": {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__min_child_weight": [1, 5, 10],
    },
}

reg_champions = {}
reg_results = []

for h in horizons:
    y_train_reg, y_test_reg, _, _ = build_targets(h)
    base_train = train_df.loc[X_train.index, base_price_col]
    base_test = test_df.loc[X_test.index, base_price_col]

    train_mask_h = y_train_reg.notna() & base_train.notna()
    test_mask_h = y_test_reg.notna() & base_test.notna()

    X_train_h = X_train.loc[train_mask_h]
    X_test_h = X_test.loc[test_mask_h]
    y_train_h = y_train_reg.loc[train_mask_h]
    y_test_h = y_test_reg.loc[test_mask_h]
    base_test_h = base_test.loc[test_mask_h]

    reg_models = {
        "RF": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
        ]),
        "XGB": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(random_state=42, n_jobs=-1, objective="reg:squarederror")),
        ]),
    }

    best_by_name = {}
    for name, model in reg_models.items():
        search = RandomizedSearchCV(
            model,
            param_distributions=reg_param_grids[name],
            n_iter=20,
            scoring=pearson_scorer,
            cv=tscv,
            random_state=42,
            n_jobs=-1,
        )
        search.fit(X_train_h, y_train_h)
        best_by_name[name] = search.best_estimator_

    best_name = None
    best_metrics = None
    best_model = None
    for name, model in best_by_name.items():
        preds = model.predict(X_test_h)
        metrics = regression_metrics(y_test_h, preds, base_test_h)
        reg_results.append({"horizon": h, "model": name, **metrics})
        if best_metrics is None or metrics["Pearson"] > best_metrics["Pearson"]:
            best_name = name
            best_metrics = metrics
            best_model = model

    top5 = top_features_from_model(
        best_model.named_steps["model"],
        X_train.columns.tolist(),
        top_k=5,
    )
    reg_champions[h] = {
        "model": best_name,
        "metrics": best_metrics,
        "top5": top5,
        "estimator": best_model,
    }

reg_results_df = pd.DataFrame(reg_results)
reg_results_df

,horizon,model,Pearson,MAE,RMSE,DA
0,1,RF,0.643034,5.978602,7.384236,0.606171
1,1,XGB,0.648067,6.128903,7.592837,0.602541
2,2,RF,0.719786,5.927528,7.237080,0.634236
3,2,XGB,0.519940,6.855500,8.356580,0.658867
4,3,RF,0.737465,6.487209,7.753863,0.633229
5,3,XGB,0.583154,6.494636,7.986386,0.678370


## 4. Entrenamiento y tuning de clasificadores por horizonte (RF/XGB)
Para cada horizonte, ejecutar RandomizedSearchCV (20 iteraciones) sobre RandomForestClassifier y XGBClassifier con scoring AUC.

In [6]:
def classification_metrics(y_true, proba, pred):
    acc = accuracy_score(y_true, pred)
    da = acc
    auc = roc_auc_score(y_true, proba) if y_true.nunique() > 1 else np.nan
    return {
        "Accuracy": float(acc),
        "DA": float(da),
        "AUC": float(auc) if auc == auc else np.nan,
    }

clf_param_grids = {
    "RF": {
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [4, 6, 8, 12, None],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", 0.6, 0.8],
    },
    "XGB": {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__min_child_weight": [1, 5, 10],
    },
}

clf_champions = {}
clf_results = []

for h in horizons:
    _, _, y_train_clf, y_test_clf = build_targets(h)
    train_mask_h = y_train_clf.notna()
    test_mask_h = y_test_clf.notna()

    X_train_h = X_train.loc[train_mask_h]
    X_test_h = X_test.loc[test_mask_h]
    y_train_h = y_train_clf.loc[train_mask_h]
    y_test_h = y_test_clf.loc[test_mask_h]

    clf_models = {
        "RF": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(random_state=42, n_jobs=-1)),
        ]),
        "XGB": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(random_state=42, n_jobs=-1, eval_metric="logloss")),
        ]),
    }

    best_by_name = {}
    for name, model in clf_models.items():
        search = RandomizedSearchCV(
            model,
            param_distributions=clf_param_grids[name],
            n_iter=20,
            scoring="roc_auc",
            cv=tscv,
            random_state=42,
            n_jobs=-1,
        )
        search.fit(X_train_h, y_train_h)
        best_by_name[name] = search.best_estimator_

    best_name = None
    best_metrics = None
    best_model = None
    for name, model in best_by_name.items():
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_test_h)[:, 1]
        else:
            proba = model.decision_function(X_test_h)
        pred = (proba >= 0.5).astype(int)
        metrics = classification_metrics(y_test_h, proba, pred)
        clf_results.append({"horizon": h, "model": name, **metrics})
        if best_metrics is None or metrics["DA"] > best_metrics["DA"]:
            best_name = name
            best_metrics = metrics
            best_model = model

    top5 = top_features_from_model(best_model.named_steps["model"], X_train.columns.tolist(), top_k=5)

    clf_champions[h] = {
        "model": best_name,
        "metrics": best_metrics,
        "top5": top5,
        "estimator": best_model,
    }

clf_results_df = pd.DataFrame(clf_results)
clf_results_df

,horizon,model,Accuracy,DA,AUC
0,1,RF,0.654567,0.654567,0.840145
1,1,XGB,0.637024,0.637024,0.774666
2,2,RF,0.724138,0.724138,0.803268
3,2,XGB,0.679371,0.679371,0.714075
4,3,RF,0.564428,0.564428,0.666751
5,3,XGB,0.644283,0.644283,0.516280


## 5. Resumen de campeones por horizonte
Construir las tablas finales de campeones con etiqueta de aptitud (DA > 0.60).

In [7]:
reg_summary_rows = []
for h in horizons:
    champ = reg_champions[h]
    metrics = champ["metrics"]
    reg_summary_rows.append({
        "horizon": h,
        "model": champ["model"],
        "Pearson": metrics["Pearson"],
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "DA": metrics["DA"],
        "Apto": metrics["DA"] > 0.60 if metrics["DA"] == metrics["DA"] else False,
        "Top5": ", ".join(champ["top5"]),
    })
reg_champions_df = pd.DataFrame(reg_summary_rows)
reg_champions_df

clf_summary_rows = []
for h in horizons:
    champ = clf_champions[h]
    metrics = champ["metrics"]
    clf_summary_rows.append({
        "horizon": h,
        "model": champ["model"],
        "Accuracy": metrics["Accuracy"],
        "DA": metrics["DA"],
        "AUC": metrics["AUC"],
        "Apto": metrics["DA"] > 0.60 if metrics["DA"] == metrics["DA"] else False,
        "Top5": ", ".join(champ["top5"]),
    })
clf_champions_df = pd.DataFrame(clf_summary_rows)
clf_champions_df

,horizon,model,Accuracy,DA,AUC,Apto,Top5
0,1,RF,0.654567,0.654567,0.840145,True,"eur_usd_lag_2, prepag1_urea 46_lag_2, wheat_in..."
1,2,RF,0.724138,0.724138,0.803268,True,"prepag1_urea 46_lag_2, eur_usd_lag_2, prepag1_..."
2,3,XGB,0.644283,0.644283,0.516280,True,"prepag1_urea 46_lag_2, fase_siembra, fase_crec..."


## 6. Reporte y export de parametros
Guardar TABLA_MAESTRA_DATAGIA.md y best_models_datagia.json.

In [10]:
report_root = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "reports"
    if candidate.exists():
        report_root = candidate
        break
if report_root is None:
    report_root = base_dir / "reports"
    report_root.mkdir(parents=True, exist_ok=True)

report_path = report_root / "TABLA_MAESTRA_DATAGIA.md"

reg_table = reg_champions_df.copy()
reg_table.insert(1, "task", "reg")
clf_table = clf_champions_df.copy()
clf_table.insert(1, "task", "clf")

report_df = pd.concat([reg_table, clf_table], ignore_index=True)
report_df = report_df.sort_values(["horizon", "task"]).reset_index(drop=True)

lines = [
    "# TABLA_MAESTRA_DATAGIA",
    "",
    "Resumen de campeones por horizonte (H1-H3).",
    "Regresion: Pearson/MAE/RMSE/DA. Clasificacion: Accuracy/DA/AUC.",
    "Apto: DA > 0.60.",
    "",
    "## Campeones (regresion)",
    reg_table.to_markdown(index=False),
    "",
    "## Campeones (clasificacion)",
    clf_table.to_markdown(index=False),
    "",
    "## Tabla unificada",
    report_df.to_markdown(index=False),
    "",
]
report_path.write_text("\n".join(lines), encoding="utf-8")
print("Reporte guardado en:", report_path.resolve())

def _jsonable(value):
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)

def serialize_model(model, model_name):
    params = {k: _jsonable(v) for k, v in model.get_params().items()}
    return {
        "model": model_name,
        "params": params,
    }

models_payload = {
    "reg": {
        str(h): serialize_model(reg_champions[h]["estimator"], reg_champions[h]["model"])
        for h in horizons
    },
    "clf": {
        str(h): serialize_model(clf_champions[h]["estimator"], clf_champions[h]["model"])
        for h in horizons
    },
}

config_root = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "config"
    if candidate.exists():
        config_root = candidate
        break
if config_root is None:
    config_root = base_dir / "config"
    config_root.mkdir(parents=True, exist_ok=True)

config_path = config_root / "best_models_datagia.json"
config_path.write_text(json.dumps(models_payload, indent=2), encoding="utf-8")
print("Model params saved to:", config_path.resolve())

Reporte guardado en: C:\Users\marco\Desktop\Repos\DATAGIA-21\notebooks\training\reports\TABLA_MAESTRA_DATAGIA.md
Model params saved to: C:\Users\marco\Desktop\Repos\DATAGIA-21\config\best_models_datagia.json
